Stratospheric ozone burdens

In [2]:
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr
from xmip.preprocessing import rename_cmip6
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import json
import cftime
import matplotlib
import warnings
import matplotlib.pyplot as plt
from Utils import get_ds_from_ceda
from Utils import weighted_annual_resample
import seaborn as sns
warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'font.size': 15})

from HiLLA_utils import get_ds, spatial_mean, set_time_to_center_of_bounds, yearly_timeseries

# get standard variable names, settings for plotting, region bounds etc. 
from config import models, regions, HiLLA_altitudes, path_archive, ens_mems_arise_UK, ens_mems_CE, colours, model_colors, model_names, lat_band_dict, UKESM_ARISE_run_IDs, UKESM_ARISE_run_dict, CESM_ARISE_run_IDs

In [3]:
## UKESM (note the combined run u-dm204dn587 is the 35 year run for 13km 60°)
ds_HiLLA_13_UK = weighted_annual_resample(get_ds(run = 'u-dm204dn587', var='03_mmr_plev', table='AERmon', zonal_mean=True).sel(time=slice(None, '2069')), var='03_mmr_plev')
ds_HiLLA_15_UK = weighted_annual_resample(get_ds(run = 'u-dn760', var='03_mmr_plev', table='AERmon', zonal_mean=True).sel(time=slice(None, '2069')), var='03_mmr_plev')

ds_HiLLA_13_CE = weighted_annual_resample(set_time_to_center_of_bounds(xr.open_dataset(path_archive+'/CESM_data/Ozone/case01_b.e21.BWSSP245cmip6.f09_g17.polar_SAI_sensitivity_60NS_13km_MAMJ_6Tg.001.cam.h0.O3.203501-206912.nc'), 'time_bnds').mean('lon').rename({'lat':'latitude'}).sel(time=slice(None, '2069')), var='O3')
ds_HiLLA_15_CE = weighted_annual_resample(set_time_to_center_of_bounds(xr.open_dataset(path_archive+'/CESM_data/Ozone/case04_b.e21.BWSSP245cmip6.f09_g17.polar_SAI_sensitivity_60NS_15km_MAMJ_6Tg.001.cam.h0.O3.203501-206912.nc'), 'time_bnds').mean('lon').rename({'lat':'latitude'}).sel(time=slice(None, '2069')), var='O3')

## combine into a dict
HiLLA_data = {'UKESM':{'13km':ds_HiLLA_13_UK, '15km':ds_HiLLA_15_UK},
              'CESM':{'13km':ds_HiLLA_13_CE, '15km':ds_HiLLA_15_CE}}


In [4]:
ds_HiLLA_13_CE

<xarray.Dataset> Size: 4MB
Dimensions:   (latitude: 192, lev: 70, time: 35)
Coordinates:
  * latitude  (latitude) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lev       (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
  * time      (time) object 280B 2035-01-01 00:00:00 ... 2069-01-01 00:00:00
Data variables:
    O3        (time, lev, latitude) float64 4MB 2.399e-12 ... 3.536e-08
    year      (time) int64 280B 2035 2036 2037 2038 2039 ... 2066 2067 2068 2069

In [5]:
os.listdir('/badc/cmip6/data/CMIP6/ScenarioMIP/NCAR/CESM2-WACCM/ssp245/r1i1p1f1/Amon/o3/gn/latest/')

['o3_Amon_CESM2-WACCM_ssp245_r1i1p1f1_gn_201501-206412.nc',
 'o3_Amon_CESM2-WACCM_ssp245_r1i1p1f1_gn_206501-210012.nc']

In [6]:
DS_ssp245_tas_UK = weighted_annual_resample(get_ds_from_ceda(model='UKESM1-0-LL', centre='MOHC', scenario_group = 'ScenarioMIP',
                              scenario = 'ssp245', variable='o3', members=ens_mems_arise_UK, 
                              table='Amon', grid='gn'), var='o3').sel(time=slice('2015', '2069')).mean('x').rename({'y':'latitude'})

DS_ssp245_tas_CE = weighted_annual_resample(get_ds_from_ceda(model='CESM2-WACCM', centre='NCAR', scenario_group = 'ScenarioMIP',
                              scenario = 'ssp245', variable='o3', members=ens_mems_CE, 
                              table='Amon', grid='gn'), var='o3').sel(time=slice('2015', '2069')).mean('x').rename({'y':'latitude'})

SSP245_data = {'UKESM':DS_ssp245_tas_UK, 'CESM':DS_ssp245_tas_CE}

OSError: no files to open